# 01 · Build RAG Index

**Phase notebook** — feature `004-notebook-split`. Produces `chroma_index/` on Google Drive.
Run top-to-bottom on a fresh Colab runtime. All constants come from the shared `config.py`.

## Deviations
- **Notebook decomposition (constitution v1.2.0 §I / §Notebook Decomposition).** One of four phase
  notebooks split from the original single `notebook.ipynb`, for modularity/reuse and dependency-conflict
  isolation. Phases hand off **only** via Drive artifacts; the single shared `config.py` is the only
  cross-notebook import. This supersedes the single-notebook risk-first stage order (recorded per §Governance).


In [ ]:
# ── Cell 1 · Bootstrap: fetch shared config.py, install THIS phase's deps ──
# The four phase notebooks share exactly ONE module (config.py), fetched from the
# repo (constitution v1.2.0 §Notebook Decomposition). Set REPO_URL to your repo.
# For a PRIVATE repo, use a token URL (https://<TOKEN>@github.com/...) or the
# raw-download fallback shown below.
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_ORG/fine-tuning-demo.git"  # TODO: set to your repo
REPO_DIR = "/content/fine-tuning-demo"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)
# Fallback (config.py only, e.g. private repo without clone access):
#   from urllib.request import urlretrieve
#   urlretrieve("<raw-url>/config.py", "/content/config.py"); sys.path.insert(0, "/content")

import config  # the ONLY cross-notebook import (FR-012)

# RAG build phase: embeddings + vector store + splitter ONLY (no training/UI stack).
# This phase's MINIMAL dependency subset (§III / FR-004) — pinned HERE, not shared.
PINNED_REQS = [
    "sentence-transformers>=5.2",
    "chromadb>=1.0",
    "langchain-text-splitters>=0.3",
    "pypdf>=4.0",
]

config.mount_drive()
config.install_deps(PINNED_REQS)
config.set_seeds()

In [ ]:
# ── Cell 2 · Build the RAG vector index (ported from monolith Stage 4) ──
import pypdf
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import chromadb

CHROMA_PATH   = config.chroma_index_path()
embed_model   = SentenceTransformer(config.EMBED_MODEL)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

_gate = config.artifact_gate("RAG index", CHROMA_PATH, "chroma.sqlite3", rebuild_verb="Rebuild")
if _gate == "skip":
    collection = chroma_client.get_collection("domain_docs")
    print(f"\u23e9 Skipping index build \u2014 loaded {collection.count()} chunks from Drive")
else:
    # Load documents
    all_texts, all_sources = [], []
    for fpath in sorted(Path(config.DOCS_PATH).rglob("*")):
        if not fpath.is_file():
            continue
        if fpath.suffix == ".pdf":
            reader = pypdf.PdfReader(str(fpath))
            text   = "\n".join(page.extract_text() or "" for page in reader.pages)
        elif fpath.suffix in (".txt", ".md"):
            text = fpath.read_text(encoding="utf-8", errors="ignore")
        else:
            continue
        if text.strip():
            all_texts.append(text)
            all_sources.append(fpath.name)
    print(f"Loaded {len(all_texts)} documents from {config.DOCS_PATH}/")
    if not all_texts:
        raise FileNotFoundError(
            f"No .pdf/.txt/.md documents found under {config.DOCS_PATH}/ \u2014 upload domain_docs first.")

    # Chunk
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP)
    chunks, metadatas = [], []
    for text, source in zip(all_texts, all_sources):
        for i, chunk in enumerate(splitter.split_text(text)):
            chunks.append(chunk)
            metadatas.append({"source": source, "chunk_id": i})
    print(f"Created {len(chunks)} chunks")

    # Embed + build collection
    embeddings = embed_model.encode(chunks, batch_size=32, show_progress_bar=True).tolist()
    try:
        chroma_client.delete_collection("domain_docs")
    except Exception:
        pass
    collection = chroma_client.create_collection("domain_docs", metadata={"hnsw:space": "cosine"})
    collection.add(
        documents=chunks, embeddings=embeddings, metadatas=metadatas,
        ids=[f"chunk_{i}" for i in range(len(chunks))])

    # Stamp the drift-detection sidecar (FR-013) LAST, after the index is complete
    config.write_meta(CHROMA_PATH, {
        "embed_model":   config.EMBED_MODEL,
        "chunk_size":    config.CHUNK_SIZE,
        "chunk_overlap": config.CHUNK_OVERLAP,
    })
    print(f"\u2705 Vector index saved to Drive: {CHROMA_PATH}  ({len(chunks)} chunks)")